---

# Curso: Data Mining - Semana 13 - Lunes (Sesión 18)

# Mapas del Perú con Folium


Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---

# <font color=blue>Notas</font>

- **Folium** crea mapas **interactivos** (zoom, popups, capas) que se pueden guardar como HTML y compartir.
- Con datos de **eventos** (cada fila = algo que ocurrió en un lugar) podemos: dibujar puntos, hacer un **mapa de calor**, **contar por región** y **agrupar** con DBSCAN.
- Para contar eventos por región usamos `sjoin`: asigna cada punto al polígono que lo contiene (la relación *within* que vimos en la diapositiva de relaciones espaciales).
- Un conteo por región se compara mejor **normalizado** (por área o por población).
- Al final hay una **plantilla** para aplicar todo esto a los datos de su proyecto.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from sklearn.cluster import DBSCAN

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
import warnings
warnings.filterwarnings("ignore")   # para evitar que muestre constantemente advertencias

## <font color=green>0. Preparamos los archivos</font>

En el panel **Archivos** de Colab necesitamos:

- la carpeta `peru_shapes` con los shapefiles de la Sesión 16;
- `sismos_peru_usgs.csv`: sismos de magnitud 4.5 o más en el Perú y alrededores, de 2000 a enero de 2024, tomados del catálogo del **USGS** (Servicio Geológico de EE. UU.).

# <font color=blue>PARTE I. Datos de eventos: sismos</font>

Cada fila es un sismo. Usaremos estas columnas:

| Columna | Significado |
|---|---|
| `time` | Fecha y hora (UTC) |
| `latitude`, `longitude` | Ubicación del epicentro (grados) |
| `depth` | Profundidad (km) |
| `mag` | Magnitud |
| `magType`| Tipo de magnitud: escala|
| `place` | Descripción del lugar (en inglés) |

## <font color=green>1. Cargamos los datos</font>

In [ ]:
sismos = pd.read_csv("sismos_peru_usgs.csv")

# La fecha viene como texto ("2007-08-15T23:40:57..."): la convertimos a fecha.
sismos["time"] = pd.to_datetime(sismos["time"])     # texto → fecha
sismos["anio"] = sismos["time"].dt.year             # año de cada sismo

print("Sismos en el archivo:", sismos.shape[0])
print("Desde", sismos["time"].min().date(), "hasta", sismos["time"].max().date())
sismos.head()

## <font color=green>2. GeoDataFrame y filtro: el Perú y su mar cercano</font>

Los datos vienen del catálogo de sismos del **USGS** (Servicio Geológico de los Estados Unidos), que registra la ubicación de cada sismo con su latitud y longitud (datos **georreferenciados**). El catálogo no se descarga por país, sino por un **rectángulo** de coordenadas: se pidió la **caja envolvente** (*bounding box*) del Perú, el mismo concepto que `bounds`. Pero un rectángulo no tiene la forma del país, así que en sus esquinas entran sismos de Ecuador, Colombia, Brasil, Bolivia y Chile.

**¿Y si nos quedamos solo con los que están dentro del territorio peruano?** Perderíamos muchos sismos importantes: casi la mitad ocurre **en el mar**, frente a la costa, donde la placa de Nazca choca con el continente (por ejemplo, el terremoto de Pisco de 2007).

La solución a este problema es sar un **buffer**, de esta forma agrandamos el contorno del Perú **150 km hacia afuera** en todas direcciones, como si lo dibujáramos con un plumón de 150 km de grosor. Esa zona ampliada incluye el territorio y el mar cercano. Después nos quedamos con los sismos que caen **dentro** (`within`) de esa zona.

- `buffer` es una **operación**: crea una geometría nueva (la zona ampliada).
- `within` es una **relación**: responde verdadero o falso para cada sismo.

Para medir 150 km necesitamos un CRS en **metros**, porque en grados no se puede medir una distancia fija: usamos `estimate_utm_crs()`, que elige la zona UTM del Perú (18S).

In [ ]:
# Puntos (X = longitud, Y = latitud) en grados
gdf = gpd.GeoDataFrame(
    sismos,
    geometry=gpd.points_from_xy(sismos["longitude"], sismos["latitude"]),
    crs="EPSG:4326"
)

# Shapefiles de la Sesión 16: contorno del país y regiones
peru = gpd.read_file("peru_shapes/per_admbnda_adm0_ign_20200714.shp")
regiones = gpd.read_file("peru_shapes/per_admbnda_adm1_ign_20200714.shp")

# 1) CRS en metros para el Perú (UTM zona 18S)
crs_metros = gdf.estimate_utm_crs() #elige el sistema de coordenadas en metros y devuelve el código EPSG.
print("CRS en metros:", crs_metros)

# 2) Buffer: el contorno del Perú agrandado 150 km (150_000 m) hacia afuera
peru_m = peru.to_crs(crs_metros) # se transforman las coordenadas de peru a metros
zona = peru_m.buffer(150_000).iloc[0]   # .iloc[0]: el shapefile tiene una sola fila (el país)

# 3) Filtro: ¿qué sismos están dentro de la zona?
todos_m = gdf.to_crs(crs_metros)        # se transforman las coordenadas de sismos a metros
dentro = todos_m.within(zona)           # ¿cada sismo está dentro de la zona? True / False para cada sismo
gdf_m = todos_m[dentro].copy()          # solo los que caen en la zona

print("Sismos en el archivo (rectángulo):", len(todos_m))
print("Sismos en el Perú y su mar cercano:", len(gdf_m))
print("Sismos descartados:", (~dentro).sum())

In [ ]:
# Gráfico del filtro: zona (buffer) de 150 km, contorno del Perú y sismos dentro / fuera del buffer

# 1) Dibujamos el buffer (la zona ampliada) en celeste.
#    zona es una sola geometría: la ponemos en una GeoSeries para poder usar .plot()
ax = gpd.GeoSeries([zona], crs=crs_metros).plot(figsize=(8, 9), color="lightblue", alpha=0.5)

# 2) Encima, el límite del Perú: solo el borde negro, sin relleno
peru_m.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1)

# 3) Los sismos dentro del buffer en rojo: son los que usaremos en el resto del laboratorio
gdf_m.plot(ax=ax, color="red", markersize=4, label="sismos dentro del buffer")

ax.set_title("Celeste: el Perú + 150 km (buffer)",          # título principal
             fontsize=14, fontweight="bold", pad=22)      # pad: deja espacio para el subtítulo
ax.text(0.5, 1.01, "Sismos M ≥ 4.5, 2000–2024 · Fuente: USGS",   # subtítulo
        transform=ax.transAxes,                           # posición relativa al mapa (0 a 1)
        ha="center", fontsize=10, color="gray")
ax.legend(loc="lower left")
#ax.set_axis_off()                                     # ocultamos los ejes con números
plt.show()

**Lectura:** la zona celeste es el buffer: el Perú + 150 km. Los puntos rojos son los **2 894 sismos** que quedan para el análisis; muchos están en el mar, frente a la costa. Los otros 258 sismos del archivo quedaron fuera del buffer (la mayoría en Ecuador) y se descartaron.

## <font color=green>3. Exploramos los atributos</font>


#### <font color="#e34a33">¿Cuántos sismos hubo cada año?</font>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.countplot(data=gdf_m, x="anio", color="steelblue", ax=ax)
ax.set_title("Sismos por año (M ≥ 4.5)")
ax.set_xlabel("")
ax.set_ylabel("sismos")
ax.tick_params(axis="x", rotation=45)
#ax.bar_label(ax.containers[0]) # nro. observaciones encima de cada barra
plt.show()

Hay dos años con picos: **2001** y **2007**. Corresponden al terremoto de **Arequipa** (23 de junio de 2001, magnitud 8.4) y al de **Pisco** (15 de agosto de 2007, magnitud 8.0): después de un gran sismo vienen cientos de **réplicas**. Lo veremos de nuevo en el mapa.

**Ojo:** la barra de **2024** casi no se ve porque el archivo llega solo hasta el 7 de enero de 2024. No es que haya habido menos sismos: faltan datos.

In [ ]:
# Resaltamos los años de los dos grandes terremotos (Arequipa 2001 y Pisco 2007).
gdf_m["tipo_anio"] = np.where(gdf_m["anio"].isin([2001, 2007]),
                              "grandes terremotos", "otros años")

fig, ax = plt.subplots(figsize=(12, 4))
sns.countplot(data=gdf_m, x="anio", hue="tipo_anio",
              #palette={"otros años": "#9ecae1", "grandes terremotos": "#D1495B"},
              dodge=False, ax=ax)
ax.set_title("Sismos por año (M ≥ 4.5)")
ax.set_xlabel("")
ax.set_ylabel("sismos")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="")
plt.show()


#### <font color="#e34a33">¿A qué profundidad ocurren?</font>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=gdf_m, x="depth", bins=40, ax=ax)
ax.set_title("Profundidad de los sismos")
ax.set_xlabel("profundidad (km)")
plt.show()

**Lectura:**
* La mayoría de los sismos son superficiales, a menos de 70 km.
* Hay otro grupo intermedio, entre 70 y 300 km.
* Hay muy pocos profundos, cerca de 600 km.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=gdf_m, x="depth", bins=40, color="#2A9D8F", edgecolor="white", ax=ax)

# Las líneas punteadas en 70 y 300 km  marcan los cortes que se usan después, en la Parte IV, para clasificar los sismos:
# superficial < 70 km, intermedio 70–300 km, profundo > 300 km
for corte in [70, 300]:
    ax.axvline(corte, color="red", linestyle="--")
ax.set_title("Profundidad de los sismos")
ax.set_xlabel("profundidad (km)")
ax.set_ylabel("sismos")
plt.show()

# <font color=blue>PARTE II. Primer mapa interactivo con Folium</font>

**Folium** es una librería para hacer **mapas interactivos**: se puede acercar, mover, hacer clic en los puntos y activar capas. Sobre el mapa podemos dibujar marcadores, círculos y otros datos.

El mapa de fondo lo da un **proveedor de mapas**, que define su estilo. Usamos el de **Esri** (la variable `MAPA_BASE` de la primera celda), porque OpenStreetMap bloquea sus mapas en Colab.

Folium trabaja con **latitud y longitud en grados** (EPSG:4326). Los cálculos los hacemos en metros, pero para dibujar volvemos a grados.

## <font color=green>4. Mapa base</font>

In [ ]:
# Mapa de fondo para todos los mapas de Folium.
# OpenStreetMap bloquea sus mapas en Colab ("Access blocked"), así que usamos los de Esri.
# Prueben cuál les gusta más (dejen solo una línea sin #):

#MAPA_BASE = "Esri.WorldTopoMap"        # colores tipo Google Maps: relieve, vegetación, ríos y vías
#MAPA_BASE = "Esri.WorldGrayCanvas"   # gris neutro: los datos resaltan más
#MAPA_BASE = "Esri.WorldImagery"      # imagen satelital
#MAPA_BASE = "Esri.WorldStreetMap"        # fondo claro, avenidas en amarillo/naranja, parques en verde
#MAPA_BASE = "OpenStreetMap.HOT"        # estilo OpenStreetMap, servido por OSM Francia
MAPA_BASE = "OpenStreetMap.France"     # estilo OpenStreetMap, servido por OSM Francia

Crear un mapa interactivo con Folium.
* `location=[latitud, longitud]` recibe las coordenadas del **centro** del mapa. ¡En Folium el orden es latitud primero (al revés que `points_from_xy`)!
* `zoom_start` define el nivel inicial de acercamiento: 5 muestra todo el Perú; 16, unas pocas cuadras.
* `tiles=MAPA_BASE` indica el mapa de fondo (lo definimos en la primera celda: Esri).
* `folium.Marker` agrega un marcador (un "pin") en un punto; `popup` es el texto que aparece al hacer clic.
* `.add_to(mapa)` coloca el elemento dentro del mapa. Si no lo agregamos, no se dibuja.
* Escribir `mapa` al final de la celda muestra el mapa en el notebook.

In [ ]:
# location (¡en Folium el orden es latitud primero!)

ubicacion_up = [-12.083915, -77.048690] # Coordenadas de la Universidad del Pacífico
mapa = folium.Map(
    location=ubicacion_up,
    zoom_start=16,
    tiles= MAPA_BASE
        )

# Un marcador de referencia: la Universidad del Pacífico
folium.Marker(
    location=ubicacion_up,
    popup="Universidad del Pacífico",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(mapa)


mapa.add_child(folium.LatLngPopup())   # al hacer clic en el mapa, muestra latitud y longitud
'''
# Polígono alrededor de la universidad ([latitud, longitud], en orden)
esquinas = [[-12.0834, -77.0488],
            [-12.0834, -77.0483],
            [-12.0840, -77.0477],
            [-12.0844, -77.0482],
            [-12.0841, -77.0488]]
folium.Polygon(
    locations=esquinas,
    color="blue",          # color del borde
    weight=2,              # grosor del borde
    fill=True,             # con relleno
    fill_opacity=0.15,     # relleno casi transparente, para ver el mapa debajo
    popup="Campus (aproximado)"
).add_to(mapa)
'''
mapa

## <font color=green>5. Los sismos más fuertes como círculos con popup</font>

Dibujamos solo los sismos de magnitud 6 o más. El tamaño del círculo crece con la magnitud y el *popup* muestra los datos al hacer clic.

Dibujar un círculo por cada sismo fuerte, con información al hacer clic.

In [ ]:
MAPA_BASE = "OpenStreetMap.HOT"

In [ ]:
# Volvemos a grados para Folium
gdf_gr = gdf_m.to_crs("EPSG:4326") #Folium dibuja en grados. Volvemos de metros a grados.
fuertes = gdf_gr[gdf_gr["mag"] >= 6]
print("Sismos de magnitud 6 o más:", len(fuertes))

mapa2 = folium.Map(
    location=[-9.2, -75.0],
    zoom_start=5,
    tiles= MAPA_BASE,
    width='80%'   #  El mapa ocupa el 80 % del ancho de la salida.
)

# Itera sobre cada fila en el DataFrame para agregar círculos y marcadores
for index, fila in fuertes.iterrows():
    # Convertimos el objeto Timestamp a texto con formato YYYY-MM-DD
    fecha_str = fila["time"].strftime("%Y-%m-%d") # fecha --> texto (año-mes-día)
    texto = (f"<b>Magnitud {fila['mag']}</b><br>"
             f"Fecha: {fecha_str}<br>"
             f"Profundidad: {fila['depth']:.0f} km<br>"
             f"{fila['place']}")

    # Dibuja un círculo en [latitud, longitud]
    folium.CircleMarker(
        location=[fila["latitude"], fila["longitude"]],
        radius=(fila["mag"] - 5) * 6,      # más magnitud = círculo más grande
        color="darkred",
        fill=True,
        fill_opacity=0.5,
        popup=folium.Popup(texto, max_width=250)
    ).add_to(mapa2)

mapa2

## <font color=green>6. Mapa de calor con todos los sismos</font>

`HeatMap` recibe una lista de pares `[latitud, longitud]`.

Crear un mapa de calor: muestra dónde hay **más sismos**, sin dibujar cada punto.
* `gdf_gr[["latitude", "longitude"]].values.tolist()`: convierte las dos columnas en una lista de pares `[latitud, longitud]`, que es lo que pide `HeatMap`.
* `radius`: radio de influencia de cada punto (en píxeles). Más grande = manchas más amplias.
* `blur`: cuánto se difumina cada mancha. Más grande = colores más suaves.
* Los colores van de azul (pocos sismos) a rojo (muchos). Es una **densidad**: no forma grupos ni marca ruido (como vimos en el Lab 1).
* Al acercar el zoom, el mapa de calor se recalcula y muestra más detalle.

In [ ]:
MAPA_BASE = "Esri.WorldStreetMap"

In [ ]:
# Lista de pares [latitud, longitud], uno por sismo: es el formato que pide HeatMap
puntos = gdf_gr[["latitude", "longitude"]].values.tolist()

mapa3 = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles= MAPA_BASE)
HeatMap(puntos, radius=8, blur=10).add_to(mapa3)
mapa3

In [ ]:
# Mapa con colores mas llamativos
puntos = gdf_gr[["latitude", "longitude"]].values.tolist()

mapa3 = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles= MAPA_BASE)

# Escala de color de naranja claro a rojo oscuro (como "OrRd" en Plotly):
# 0 = poca densidad, 1 = mucha densidad --> cada número (de 0 a 1) es un nivel de densidad
#  y cada color, el que se usa en ese nivel.
rojos = {0.2: "#fdbb84", 0.4: "#fc8d59", 0.6:"#e34a33" , 0.8:"#b30000" , 1.0:"#7f0000" }

HeatMap(puntos, radius=8, blur=10,
        gradient=rojos,       # nuestros colores, en lugar de azul → verde → rojo
        min_opacity=0.3       # las zonas con poca densidad se ven más suaves
        ).add_to(mapa3)

mapa3

### <font color=green>6.1. Todos los sismos agrupados con MarkerCluster</font>

`MarkerCluster` agrupa solo para dibujar pero no es un clustering

In [ ]:
from folium.plugins import MarkerCluster

mapa4 = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles=MAPA_BASE)

grupo = MarkerCluster().add_to(mapa4)          # el "contenedor" que agrupa los puntos
for _, fila in gdf_gr.iterrows():
    folium.Marker(
        location=[fila["latitude"], fila["longitude"]],
        popup=f"M {fila['mag']} · {fila['time'].date()}"
    ).add_to(grupo)                           # se agregan al cluster, no al mapa

mapa4

# <font color=blue>PARTE III. Contar por región: mapa coroplético</font>

## <font color=green>7. Asignamos cada sismo a su región con `sjoin`</font>
**Objetivo:** saber en qué región ocurrió cada sismo, para después contar cuántos sismos tiene cada región.

**El problema:** la tabla de sismos solo tiene coordenadas (latitud y longitud), no tiene una columna con la región. Por eso no podemos usar `merge`, que necesita una columna en común.

**La solución: `sjoin`** (*spatial join*, unión espacial). Une la tabla de sismos con la de regiones **por su ubicación**: revisa en qué polígono cae cada sismo (la relación `within`) y le agrega el nombre y el código de esa región.

**Resultado:** la misma tabla de sismos, con dos columnas nuevas (`ADM1_ES` y `ADM1_PCODE`). Los sismos que ocurrieron en el mar no caen en ninguna región y quedan con `NaN`.

#### <font color="#e34a33">Asignar a cada sismo la región donde ocurrió.</font>

In [ ]:
regiones_m = regiones.to_crs(crs_metros)

# Unimos Lima y Callao: el Callao es tan pequeño que su tasa por área no sería confiable
regiones_m["ADM1_ES"] = regiones_m["ADM1_ES"].replace({"Lima": "Lima y Callao", "Callao": "Lima y Callao"})
regiones_m = regiones_m.dissolve(by="ADM1_ES", as_index=False)   # un solo polígono para las dos
print("Regiones:", len(regiones_m))                              # 24 (eran 25)

sismos_region = gpd.sjoin(
    gdf_m,
    regiones_m[["ADM1_ES", "ADM1_PCODE", "geometry"]], # De las regiones traemos el nombre, el código y la geometría.
    how="left",            # conserva todos los sismos
    predicate="within"     # el punto está dentro del polígono
)

# Los sismos en el mar no caen en ninguna región: quedan con NaN.
print("Sismos en alguna región:", sismos_region["ADM1_ES"].notna().sum())
print("Sismos en el mar (sin región):", sismos_region["ADM1_ES"].isna().sum())

## <font color=green>8. Conteo y normalización por área</font>

Una región grande tiene más espacio para que ocurran sismos. Para comparar de forma justa calculamos **sismos por cada 10 000 km²**.

Contar los sismos de cada región y dividir por su área.
* `groupby("ADM1_ES").size()`: cuenta cuántos sismos tiene cada región.
* `regiones_m.area / 1_000_000`: el área sale en m² (porque el CRS está en metros); dividimos entre un millón para obtener km².
* `merge(conteo, on="ADM1_ES", how="left")`: pega el conteo a la tabla de regiones. Con `how="left"` se conservan todas las regiones.
* `fillna(0)`: las regiones sin sismos quedaban vacías (`NaN`); les ponemos 0.
* `sismos / area_km2 * 10_000`: la **tasa** de sismos por cada 10 000 km², para comparar regiones de distinto tamaño.

#### <font color="#e34a33">Contar los sismos de cada región y dividir por su área.</font>

In [ ]:
# Cuenta cuántos sismos tiene cada región.
conteo = sismos_region.groupby("ADM1_ES").size().rename("n_sismos").reset_index()

# Área de cada región en km²
regiones_m["area_km2"] = regiones_m.area / 1_000_000  # de metros m² a km²

# Unimos el conteo al mapa (merge, como en la Sesión 16)
regiones_conteo = regiones_m.merge(conteo, on="ADM1_ES", how="left")

#Si una región no tuvo sismos, el conteo correcto es 0
regiones_conteo["n_sismos"] = regiones_conteo["n_sismos"].fillna(0).astype(int)

# Tasa: sismos por cada 10 000 km², para comparar regiones de distinto tamaño.
# Dividimos los sismos entre el área (sismos por km²) y multiplicamos por 10 000
# para que el número sea más fácil de leer (Tacna: 0.0056 → 56). Redondeamos a 1 decimal.
regiones_conteo["sismos_10mil_km2"] = (
    regiones_conteo["n_sismos"] / regiones_conteo["area_km2"] * 10_000
).round(1)

display(
    regiones_conteo[["ADM1_ES", "n_sismos", "area_km2", "sismos_10mil_km2"]]
    .sort_values("sismos_10mil_km2", ascending=False).head(10).round(0)
)

In [ ]:
# Las 15 regiones con más sismos por 10 000 km² (seaborn)
top = regiones_conteo.sort_values("sismos_10mil_km2", ascending=False).head(15)
ax = sns.barplot(data=top, x="sismos_10mil_km2", y="ADM1_ES",
                 hue="sismos_10mil_km2", palette='rocket_r', legend=False) #'hsv' "tab20"
ax.set_title("Sismos por cada 10 000 km²")
ax.set_xlabel("sismos por 10 000 km²")
ax.set_ylabel("")
plt.show()

## <font color=green>9. Mapa coroplético interactivo con `explore()`</font>

`explore()` de GeoPandas crea un mapa de Folium en una sola línea. Pasar el mouse sobre una región para ver sus datos.

#### <font color="#e34a33">Dibujar un mapa coroplético (regiones coloreadas según un valor).</font>
`simplify(1000)`: simplifica los bordes con una tolerancia de 1 000 m. El mapa carga más rápido y la forma general no cambia.



In [ ]:
# Simplificamos los límites para que el mapa cargue rápido
# (tolerancia de 1 km; la forma general no cambia).
regiones_mapa = regiones_conteo.copy()
regiones_mapa["geometry"] = regiones_mapa.simplify(1000) #hace que los bordes tengan menos puntos.

# explore: crea un mapa Folium
mapa5 = regiones_mapa.explore(
    column="sismos_10mil_km2", # Columna que define el color de cada región.
    cmap="rocket_r",     # paleta de seaborn: más oscuro = más sismos
    tooltip=["ADM1_ES", "n_sismos", "sismos_10mil_km2"], #columnas que se muestran al pasar el mouse sobre la región
    legend=True,
    tiles= MAPA_BASE,
    style_kwds={"color": "gray", "weight": 0.5}
)
mapa5

**Lectura:**

- Las regiones del **sur** (Tacna, Arequipa, Moquegua) y de la **costa central** (Ica) concentran más sismos por km².
- Los sismos **en el mar** no aparecen en este mapa porque no caen en ninguna región. El mapa de calor sí los mostraba.

# <font color=blue>PARTE IV. Un atributo en el mapa: la profundidad</font>

Clasificamos cada sismo por su profundidad con los rangos que usan los sismólogos y pintamos el mapa por esa clase. Es la idea del Lab 1 (Canadá): primero un atributo, después vemos **dónde** queda cada grupo.

## <font color=green>10. Tres clases de profundidad</font>

#### <font color="#e34a33">Clasificar cada sismo por su profundidad y pintar el mapa por clase.</font>


In [ ]:
# cut: discretiza una variable numérica en categorías por rangos intercuatílicos
#    bins=[0, 70, 300, 800]: los cortes (0–70, 70–300 y 300–800 km).
#    labels: el nombre de cada rango. Llevan número adelante para que se ordenen bien en la leyenda.
gdf_m["tipo_profundidad"] = pd.cut(
    gdf_m["depth"],
    bins=[0, 70, 300, 800],
    labels=["1. Superficial (< 70 km)", "2. Intermedio (70–300 km)", "3. Profundo (> 300 km)"]
)
print(gdf_m["tipo_profundidad"].value_counts().sort_index())

mapa6 = gdf_m.to_crs("EPSG:4326").explore(
    column="tipo_profundidad",
    categorical=True,
    cmap=["#f4a261", "#2a9d8f", "#264653"],   # un color por clase
    marker_kwds={"radius": 3},
    tooltip=["time", "mag", "depth", "place"],
    tiles= MAPA_BASE,
    legend=True
)
mapa6

### Interpretación

- **Superficiales**: frente a la costa y en la costa, donde la placa de Nazca empieza a hundirse bajo la placa Sudamericana.
- **Intermedios**: debajo de los Andes.
- **Profundos**: en el este, cerca de la frontera con Brasil.

La profundidad aumenta de oeste a este: el mapa muestra cómo la placa de Nazca se hunde bajo el continente. El atributo, visto en el mapa, revela una estructura espacial.

# <font color=blue>PARTE V. DBSCAN espacial en el mapa interactivo</font>

## <font color=green>11. DBSCAN con eps = 30 km</font>

Igual que en el Lab 1: coordenadas en **metros** y `eps` en metros.

Aplicar DBSCAN a las ubicaciones de los sismos.

In [ ]:
# column_stack([x, y]): arma una matriz de coordenadas (una fila por sismo, dos columnas en metros).
coords_m = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y])

gdf_m["cluster"] = DBSCAN(eps=30_000, min_samples=20).fit_predict(coords_m)

# cuántos grupos hay, sin contar el ruido.
n_grupos = gdf_m["cluster"].nunique() - (1 if -1 in gdf_m["cluster"].values else 0)
print("Grupos:", n_grupos)
print("Ruido:", (gdf_m["cluster"] == -1).sum())

Perfilar cada grupo: ¿cuántos sismos tiene, a qué profundidad, en qué año y cuál fue el más fuerte?

In [ ]:
# Perfil de cada grupo: tamaño, profundidad, año más frecuente y sismo más fuerte
perfil = (
    gdf_m[gdf_m["cluster"] != -1] # dejamos fuera el ruido.
    .groupby("cluster")
    #  calcula varios resúmenes a la vez y le pone nombre a cada uno
    .agg(   # nombre=("columna", función)
         sismos=("mag", "size"), #cuántos sismos
         prof_mediana_km=("depth", "median"),
         anio_mas_frecuente=("anio", lambda s: s.mode()[0]), # valor más frecuente (la moda)
         mag_maxima=("mag", "max"))
    .sort_values("sismos", ascending=False)
)
perfil.head(8)

#### <font color="#e34a33">Dibujar los grupos de DBSCAN en un mapa interactivo, con el ruido en gris.</font>


In [ ]:
# Mapa interactivo: un color por grupo, ruido en gris
gdf_c = gdf_m.to_crs("EPSG:4326")

# convertimos el número de grupo en texto para tratarlo como categoría; el -1 pasa a llamarse "ruido".
gdf_c["grupo"] = gdf_c["cluster"].astype(str)
gdf_c.loc[gdf_c["cluster"] == -1, "grupo"] = "ruido"

# Primero dibujamos el ruido
mapa_db = gdf_c[gdf_c["grupo"] == "ruido"].explore(
    color="lightgray", marker_kwds={"radius": 2},
    tiles= MAPA_BASE, name="ruido"
)

# Luego se dibuja los grupos sobre el mismo mapa
gdf_c[gdf_c["grupo"] != "ruido"].explore(
    m=mapa_db,                          # dibujamos sobre el mismo mapa
    column="grupo", categorical=True, cmap="tab20",
    marker_kwds={"radius": 3},
    tooltip=["grupo", "time", "mag", "depth"],
    legend=False, name="grupos DBSCAN"
)
mapa_db

### Lectura:

- El grupo de **Ica** tiene muchos sismos en el **2007**: son réplicas del terremoto de Pisco.
- Varios grupos de **Arequipa** se concentran en **2001**: réplicas del terremoto de Arequipa.
- **Limitación:** DBSCAN agrupa por cercanía en el **espacio**, pero algunos grupos son en realidad **secuencias en el tiempo** (un gran sismo y sus réplicas). Antes de interpretar un clúster, revisen también cuándo ocurrió.
- Con `eps` mayor (50 km), DBSCAN encadena toda la franja sísmica del sur en un solo grupo: los sismos forman una banda continua.

# <font color=blue>PARTE VI. Mapa final con capas</font>

## <font color=green>12. Un solo mapa con capas que se activan y desactivan</font>

`folium.FeatureGroup` agrupa elementos en una capa; `LayerControl` agrega el selector de capas (arriba a la derecha).

Construir un solo mapa con tres capas que se pueden activar y desactivar.

**Capa 1 · `folium.Choropleth`**: otra forma de hacer un mapa coroplético, directamente en Folium.

**Capas 2 y 3 · `folium.FeatureGroup`**: una carpeta donde guardamos elementos (el mapa de calor, los círculos) para tratarlos como una sola capa.

* `show=False`: la capa empieza **apagada**; el usuario la enciende en el selector.

**`folium.LayerControl()`**: agrega el selector de capas (arriba a la derecha). Va **al final**, cuando todas las capas ya están en el mapa.

**`mapa_final.save("mapa_sismos_peru.html")`**: guarda el mapa como un archivo HTML. Se abre en cualquier navegador, sin Python, y se puede adjuntar al informe del proyecto (lo encuentran en el panel Archivos de Colab).

In [ ]:
mapa_final = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles=MAPA_BASE) #[-9.2, -75.0] es el centro del Perú,

# Capa 1: coroplético de sismos por 10 000 km²
folium.Choropleth(
    geo_data=regiones_mapa.to_crs("EPSG:4326")[["ADM1_ES", "sismos_10mil_km2", "geometry"]], #en grados
    data=regiones_mapa, # la tabla con los valores a pintar
    columns=["ADM1_ES", "sismos_10mil_km2"], #  región y valor.
    key_on="feature.properties.ADM1_ES", #  es como el on= de un merge.
    fill_color="OrRd",
    fill_opacity=0.6,
    line_weight=0.5,  #  grosor del borde
    legend_name="Sismos por 10 000 km²",
    name="Sismos por región" # nombre de la capa
).add_to(mapa_final)

# Capa 2: mapa de calor
capa_calor = folium.FeatureGroup(name="Mapa de calor", show=False) # la capa empieza apagada
HeatMap(puntos, radius=8, blur=10).add_to(capa_calor)
capa_calor.add_to(mapa_final)

# Capa 3: sismos fuertes con popup
capa_fuertes = folium.FeatureGroup(name="Sismos M ≥ 6") # nombre de la capa
for _, fila in fuertes.iterrows():
    # Formateamos el objeto Timestamp a texto con strftime
    fecha_str = fila["time"].strftime("%Y-%m-%d")
    folium.CircleMarker(
        location=[fila["latitude"], fila["longitude"]],
        # Fórmula para que el tamaño del círculo dependa de la magnitud, y que la diferencia entre sismos se note a simple vista
        radius=(fila["mag"] - 5) * 6, color="darkred", fill=True, fill_opacity=0.5, # radio en píxeles
        popup=folium.Popup(f"<b>M {fila['mag']}</b> · {fecha_str}<br>{fila['place']}", max_width=250)
    ).add_to(capa_fuertes)
capa_fuertes.add_to(mapa_final)

folium.LayerControl().add_to(mapa_final) # agrega al mapa el selector de capas

# Guardamos el mapa como HTML: se abre en cualquier navegador y se puede adjuntar al informe.
mapa_final.save("mapa_sismos_peru.html")
mapa_final

# <font color=blue>PARTE VII. Plantilla para su proyecto</font>

Copien la celda que corresponda a sus datos y cambien solo las variables en MAYÚSCULAS.

## <font color=green>13. Caso A: mi tabla tiene latitud y longitud</font>

Ejemplos: clientes, tiendas, sensores, reclamos, incidentes.

Cómo usar la plantilla A:
* Cambien solo las variables en MAYÚSCULAS: el archivo y los nombres de sus columnas.
* `dropna(subset=[COL_LAT, COL_LON])`: elimina las filas sin coordenadas.
* `location=[media de latitud, media de longitud]`: el mapa se centra solo en sus datos.
* El mapa de calor usa **todos** los puntos; los círculos usan una **muestra** de `MAX_PUNTOS` (`sample`), porque miles de círculos hacen que el mapa sea muy lento.
* `random_state=1`: la muestra es siempre la misma cada vez que corren la celda.
* `popup=str(fila[COL_INFO])`: el texto que aparece al hacer clic en cada punto.

In [ ]:
# ---- CAMBIEN ESTO ----
ARCHIVO = "sismos_peru_usgs.csv"   # su archivo
COL_LAT = "latitude"               # columna de latitud
COL_LON = "longitude"              # columna de longitud
COL_INFO = "place"                 # columna que se mostrará al hacer clic

MAX_PUNTOS = 1000                  # para que el mapa no se vuelva lento
# ----------------------

datos = pd.read_csv(ARCHIVO).dropna(subset=[COL_LAT, COL_LON])
print("Filas con coordenadas:", len(datos))

m = folium.Map(location=[datos[COL_LAT].mean(), datos[COL_LON].mean()],
               zoom_start=6, tiles= MAPA_BASE)

HeatMap(datos[[COL_LAT, COL_LON]].values.tolist(), radius=8, blur=10,
        name="Mapa de calor").add_to(m)

puntos_capa = folium.FeatureGroup(name="Puntos", show=False)
# Si hay muchos puntos, dibujamos una muestra (el mapa de calor sí usa todos).
n = min(MAX_PUNTOS, len(datos))              # si hay menos filas que el límite, usamos todas
muestra = datos.sample(n, random_state=1)    # random_state: la misma muestra cada vez
for _, fila in muestra.iterrows():
    folium.CircleMarker([fila[COL_LAT], fila[COL_LON]], radius=1, color="red",
                        popup=str(fila[COL_INFO])).add_to(puntos_capa)
puntos_capa.add_to(m)

folium.LayerControl().add_to(m)
m.save("mapa_proyecto.html")
m

## <font color=green>14. Caso B: mi tabla tiene región, provincia o distrito</font>

Usen el **código** del shapefile para unir (como en la Sesión 16). Si su tabla tiene el **ubigeo** del INEI, el código del shapefile es `"PE"` + ubigeo: por ejemplo, el ubigeo `150122` (Miraflores) corresponde a `PE150122` en `ADM3_PCODE`.

Ejemplo de fuente: el **RENADEF** del INEI publica denuncias por comisaría con su ubigeo, que se pueden agregar por distrito.

Cómo usar la plantilla B:
* `tabla`: su tabla con un código de región y un valor (aquí es un ejemplo inventado; en su proyecto usarán `pd.read_csv(...)`).
* `regiones.merge(tabla, on=COL_CODIGO, how="left")`: une por **código**, no por nombre (los nombres pueden estar escritos distinto: "Cusco" o "Cuzco").
* `simplify(1000)` en metros y de vuelta a grados: bordes más simples para que el mapa cargue rápido.
* `missing_kwds`: cómo pintar las regiones **sin dato** (en gris, con la etiqueta "sin dato").
* Para provincias o distritos, cambien `regiones` por el shapefile `adm2` o `adm3` y el código por `ADM2_PCODE` o `ADM3_PCODE`.

In [ ]:
# ---- CAMBIEN ESTO ----
# Tabla de ejemplo: un indicador para algunas regiones (en su proyecto: pd.read_csv(...))
tabla = pd.DataFrame({
    "ADM1_PCODE": ["PE15", "PE04", "PE08", "PE20", "PE16"],   # Lima, Arequipa, Cusco, Piura, Loreto
    "valor": [80, 55, 70, 45, 35]
})
COL_CODIGO = "ADM1_PCODE"
COL_VALOR = "valor"
# ----------------------

mapa_tabla = regiones.merge(tabla, on=COL_CODIGO, how="left")
mapa_tabla["geometry"] = mapa_tabla.to_crs(crs_metros).simplify(1000).to_crs(regiones.crs)

mapa_tabla.explore(
    column=COL_VALOR, cmap="rocket_r", legend=True,
    tooltip=["ADM1_ES", COL_VALOR], tiles= MAPA_BASE,
    missing_kwds={"color": "lightgray", "label": "sin dato"},
    style_kwds={"color": "gray", "weight": 0.5}
)

# OPCIONAL
# <font color=blue>PARTE VIII. Los mismos mapas con Plotly Express</font>

**Plotly Express** (`px`) también hace mapas interactivos, con el mismo estilo que sus otros gráficos (los que ya usan en el curso). Hacemos tres de los mapas anteriores en una línea cada uno:

| Mapa | Folium | Plotly Express |
|---|---|---|
| Puntos | `folium.CircleMarker` (con un `for`) | `px.scatter_map` |
| Mapa de calor | `HeatMap` | `px.density_map` |
| Coroplético | `folium.Choropleth` / `explore()` | `px.choropleth_map` |

¿Cuál usar? **Folium** es mejor para mapas con varias **capas** que se activan y desactivan, y para guardar un mapa HTML. **Plotly** es más rápido de escribir y mantiene el mismo estilo que sus otros gráficos. Los dos sirven para el proyecto.

## <font color=green>15. Preparamos Plotly y el mapa de fondo</font>

Igual que Folium, Plotly necesita un mapa de fondo. Usamos el **mismo de Esri** para evitar el bloqueo de OpenStreetMap en Colab.
* `map_style="white-bg"`: le decimos a Plotly que no use su fondo por defecto.
* `map_layers`: agregamos el fondo de Esri como una capa de imágenes (`raster`).
* `below="traces"`: el fondo va **debajo** de los datos.
* Guardamos todo en la función `fondo_esri(fig)` para aplicarla a cada mapa con una sola línea.

Las funciones `px.scatter_map`, `px.density_map` y `px.choropleth_map` necesitan Plotly 5.24 o más reciente (Colab ya lo trae).

In [ ]:
import plotly.express as px

URL_ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"

def fondo_esri(fig):
    """Pone el mapa de Esri como fondo de un mapa de Plotly."""
    fig.update_layout(
        map_style="white-bg",
        map_layers=[{"below": "traces", "sourcetype": "raster",
                     "sourceattribution": "Tiles © Esri", "source": [URL_ESRI]}],
        margin={"r": 0, "t": 40, "l": 0, "b": 0}      # sin márgenes blancos alrededor
    )
    return fig

## <font color=green>16. Puntos: los sismos más fuertes con `px.scatter_map`</font>

El mismo mapa de la sección 5, pero sin escribir un `for`: Plotly dibuja todos los puntos de la tabla a la vez.
* `lat` y `lon`: los **nombres de las columnas** con la latitud y la longitud (no hace falta armar pares como en Folium).
* `color="mag"`: el color depende de la magnitud; `color_continuous_scale` es la paleta. Con `OrRd[3:]` quitamos los 3 tonos más claros de la paleta, para que los sismos de magnitud 6 no salgan casi blancos.
* `size="mag"`: el tamaño también depende de la magnitud.
* `hover_name`: el título que aparece al **pasar el mouse**; `hover_data`: las columnas que se muestran debajo.
* `center` y `zoom`: el centro y el acercamiento inicial (como `location` y `zoom_start` en Folium).

In [ ]:
fig = px.scatter_map(
    fuertes,                                   # los sismos de magnitud 6 o más (sección 5)
    lat="latitude", lon="longitude",
    color="mag", size="mag",
    color_continuous_scale=px.colors.sequential.OrRd[3:],   # sin los tonos más claros: se verían blancos
    hover_name="place",
    hover_data={"mag": True, "depth": True, "anio": True, "latitude": False, "longitude": False},
    center={"lat": -9.2, "lon": -75.0}, zoom=4, height=600,
    title="Sismos de magnitud 6 o más"
)
fondo_esri(fig).show()

## <font color=green>17. Mapa de calor con `px.density_map`</font>

El mismo mapa de la sección 6.
* `radius`: radio de influencia de cada sismo (como en `HeatMap`).
* `range_color=[0, 5]`: el rango de la escala de color. Sin él, Plotly usa de 0 a 1 y casi todo el mapa sale rojo oscuro (saturado). Si su mapa de calor se ve todo oscuro, suban el máximo; si se ve muy pálido, bájenlo.
* `color_continuous_scale`: la paleta; más oscuro = más sismos.
* `hover_data`: aquí la dejamos vacía, porque en un mapa de calor no interesa cada punto sino la densidad.

In [ ]:
fig = px.density_map(
    gdf_4326,                                  # todos los sismos, en grados
    lat="latitude", lon="longitude",
    radius=8,
    range_color=[0, 5],                        # escala de color: de 0 a 5 sismos superpuestos
    color_continuous_scale="OrRd",
    center={"lat": -9.2, "lon": -75.0}, zoom=4, height=600,
    title="Mapa de calor de los sismos"
)
fondo_esri(fig).show()

## <font color=green>18. Mapa coroplético con `px.choropleth_map`</font>

El mismo mapa de la sección 9: regiones coloreadas por sismos cada 10 000 km².
* `geojson=reg.__geo_interface__`: las **geometrías** de las regiones, en el formato que entiende Plotly (GeoJSON).
* `locations="ADM1_ES"`: la columna de la tabla que identifica cada región.
* `featureidkey="properties.ADM1_ES"`: dónde está ese mismo identificador dentro del GeoJSON. Es la misma idea que `key_on` en `folium.Choropleth`: así Plotly sabe qué valor va en qué región.
* `color`: la columna que define el color; `opacity`: la transparencia de las regiones.

In [ ]:
# Regiones simplificadas (sección 9), en grados y solo con las columnas que necesitamos
reg = regiones_mapa.to_crs("EPSG:4326")[["ADM1_ES", "sismos", "sismos_10mil_km2", "geometry"]]

fig = px.choropleth_map(
    reg,
    geojson=reg.__geo_interface__,
    locations="ADM1_ES",
    featureidkey="properties.ADM1_ES",
    color="sismos_10mil_km2",
    color_continuous_scale="OrRd",
    hover_name="ADM1_ES",
    hover_data={"sismos": True, "sismos_10mil_km2": True, "ADM1_ES": False},
    opacity=0.7,
    center={"lat": -9.2, "lon": -75.0}, zoom=4, height=600,
    title="Sismos por cada 10 000 km²"
)
fondo_esri(fig).show()

### Para su proyecto

- Con Plotly, el mapa también se puede guardar como HTML: `fig.write_html("mi_mapa.html")`.
- Si su tabla tiene latitud y longitud, cambien la tabla y los nombres de las columnas en `px.scatter_map` o `px.density_map`.
- Si su tabla tiene región o distrito, unan primero con el shapefile por código (plantilla B) y usen `px.choropleth_map` con `locations` y `featureidkey`.

# <font color=blue>DESAFÍO</font>

## <font color=green>Proyecto</font>

Apliquen la plantilla de la Parte VII a los datos de su proyecto (caso A o caso B), guarden el HTML y escriban dos líneas sobre lo que muestra el mapa y una limitación.